# Birth-Death Population Dynamics: A Stochastic Model

Many processes in physics, biology, and other fields are characterized by some element of randomness: the timing of a nuclear decay, the trajectory of a particle undergoing Brownian motion, and even the production of proteins in a cell, for example. These are called **stochastic** processes. Rather than try to model the details of the dynamics, it is often easier to zoom out to a coarser, "meso-scopic" model based in random numbers. 

For this exercise, we'll look at a simple model of population dynamics called a **birth-death processes**. In this model, we will assume that the birth rate $\beta_\text{birth}$ of some population is a constant, while the death rate $\beta_\text{death}$ depends linearly on the size of the population $\beta_\text{death}=kP$, where $k$ is a proportionality constant and $P$ is the population.  Both birth and death rates have units of [events/time].  The total rate of events (birth or death) $\beta_\text{total}$ is given by $\beta_\text{total} = \beta_\text{birth} + \beta_\text{death}$.

We'll keep our model simple; More complicated models might include birth and death rates that depend on the size of a secondary population, such as in wolf/rabbit predation models; or they might include additional kinds of events, such as the alternate decay paths for some nuclear decays or the on/off switching of an enzyme that alters the birth rate.

To make this model stochastic, we'll let both the timing and the sequencing of individual births and deaths be probabilistic. This echos many physical and biological processes such as nuclear decay, protein synthesis, and predation where quantum, thermal, or social effects prevent us from precisely predicting individual events. 

**In this exercise set, you will build a computational model of this simple birth-death process and analyze the results.**

## 1) Theoretical prediction

Let's assume that the population  is deterministic, continuously varing (i.e. events are not discrete occurances), and continuously variable (i.e. the population  is not restricted to integers).  In this case, the rate of change of the population size is given by $$\frac{dP}{dt} = \beta_\text{birth} - \beta_\text{death}.\tag{1}$$  Eventually, the population size will approach some constant $\overline{P}$.  Using Equation 1, solve for the value of this steady state population in terms of model parameters.

By solving Equation 1 analytically, show that $$P(t) = P_0e^{-kt} + \overline{P}(1-e^{-kt})\tag{2},$$ where $P_0$ is the initial population.

## Computational solution - the Gillespie algorithm

Now, let's model the system computationally.  Computational approaches to modeling stochastic systems often use **Monte-Carlo** algorithms, which involve generating and manipulating random numbers. In order to model a stochastic birth-death process, we'll turn to a particular Monte-Carlo algorithm called the **Gillespie Algorithm** which abstracts the process into a series of steps that can be calculated numerically.

The basic algorithm is as follows:
1. Calculate the mean rate at which events (any event) occur
2. Calculate the time interval until the next event (based on a random number)
3. Determine which event actually occurs (based on a second random number)
4. Adjust population and event rate based on the outcomes of steps 2 and 3
5. Repeat

### 2) Random time intervals

Like the clicks of a Geiger counter, events in birth-death processes do not occur at regular time intervals; instead we can only say that the *odds* of an event (birth or death) occurring within any small time interval $\Delta t$ is equal to $\beta \Delta t$, where $\beta$ is the mean rate of event occurrences (and $1/\beta$ is the average time between events). This is an example of what is called a "**Poisson Process**."

Although the time intervals between sequential events fluctuate at random, they should follow an exponential distribution where the probability of obtaining a value of $t_\text{w}$ (give-or-take $\Delta t$) is given by:

$$
\mathcal{P}(t_\text{w}) = \beta e^{-\beta t_\text{w}}\tag{1}
$$

Python's `random()` function provides a *uniform* sequence of pseudorandom numbers in the range $[0,1)$. To generate a sequence of time intervals which follows Equation 1, we can convert a uniform distribution to an expontial distribution using the equation $$t_\text{w}^{\prime} = -(1/\beta)*\log(t_\text{w}).\tag{2}$$

Write a function `getTimes(beta)` that takes the mean rate $\beta$ as argument, and returns a pseudorandom time interval drawn from an exponential distribution.

Using this function, generate one thousand random time intervals (with $\beta=0.5$ s$^{-1}$), make a histogram of the values (including error bars) with a logarithmic $y$-axis, and perform an exponential fit to verify that the distribution has the intended value of $\beta$.

Equation 1 describes the probability of observing a particular time interval, and therefore has an integral of one.  The integral of your histogram is the number of intervals generated.  So be sure to allow the normalization of your fit function to float freely.

In an experiment like this, we can assume that the statistical uncertainty on $N$ counts is $\sqrt{N}$.  However, this breaks down for zero counts.  In this case, assume that the statistical uncertainty is 1.14.

Within its uncertainty, is the fitted value of $\beta$ consistent with 0.5 s$^{-1}$?

### 3) Selecting between birth/death

So far, we have a method to calculate a time interval until the next event (birth or death).  Now, we need a way to determine if the event is a birth or a death - something like a coin-flip but for which the relative odds may not be 50/50. In general, a process that randomly decides between two events with odds $\xi_1$ and $\xi_2 = 1-\xi_1$, is called a **Bernoulli Trial**. For a fair coin-flip, $\xi_1 = \xi_2 = 1/2$.

**Create a `BernoulliTrial(Xi)` function that takes the probability $\xi_1$ as an argument and randomly picks between two events.** This function should return `True` or `False` to indicate whether or not event 1 occurred.

### 4) Building the dynamics

Given the rates at which births and deaths occur ($\beta_\text{birth}$ and $\beta_\text{death}$), as well as the total rate for which any event occurs $\beta_\text{total}$, the probability that a particular event will be a birth or death is given by
\begin{align}
    \xi_\text{birth} = \dfrac{\beta_\text{birth}}{\beta_\text{total}} = \dfrac{\beta_\text{birth}}{\beta_\text{birth} + kP} \\
    \xi_\text{death} = \dfrac{\beta_\text{death}}{\beta_\text{total}} = \dfrac{kP}{\beta_\text{birth} + kP}.\\
\end{align}
As they should, these probabilities sum to one.

Now, we can put everything together. Inside of a loop, you should:
* Calculate the total rate of event occurrences (this must be inside the loop because the value changes as the population evolves)
* Randomly determine the time until the next event
* Randomly determine the type of event (birth or death)
* Update the population accordingly

Assume an initial population of 100, $k = 0.001$ s$^{-1}$, and $\beta_{\text{birth}}=0.2$ s$^{-1}$.  Simulate the system over 5,000 events.  Plot the population as a function of time, and compare to the theoretical expectation from Equation 2.

How do the result of the stochastic model compare to the analytical solution?  How do the predictions differ? Do both predictions approach $\overline{P} = \frac{\beta_\text{birth}}{k}$ as $t\rightarrow\infty$?

If you execute the code multiple times without changing any parameters, are the results repeatable?

If not, modify your code so that the results of the stochastic model are repeatable/reproducible.

### 5) Explore the model

Now let's vary the model parameters and repeat the calculation to determine the effect of each variation.  For each parameter variation, use the nominal values of all other parameters (from above), unless instructed otherwise.

We'll start by varying the initial population. Try values of 20, 200, and 2000.  What effect does this variation have on the simulation?  Does the stochastic model always match the analytic solution (modulo statistical fluctuations)?

Next, vary the value of $\beta_{\text{birth}}$.  Try values of 0.05, 0.2, and 0.8. What effect does this variation have on the simulation? Does the stochastic model always match the analytic solution (modulo statistical fluctuations)?

Finally, vary the value of $k$.  Try each order of magnitude between $10^{-5}$ and $10^{-2}$ and simulate 20,000 events. What effect does this variation have on the simulation? Does the stochastic model always match the analytic result (modulo statistical fluctuations)?

### 6) Noise and probability distributions

Noise is an inherent part of stochastic systems; even once the system reaches a steady state, the population still fluctuates about the deterministic prediction. Because of this, it is common to talk about the *population mean*, *variance*, and more generally, the *probability distribution* of the population rather than the population dynamics themselves. The probability distribution $\mathcal{P}(P,t)$ tells us the probability of a population having $P$ members at time $t$, but what does this distribution look like? 

In order to study the steady state of the model:
- Start the simulation with the population already at the steady-state value.
- Save a collection of the population values calculated after every event.
    - Calculate the mean and variance of the population values.  The variance is the expected value of the squared deviation from the mean, $\sigma^2 = \frac{1}{N}\sum_{i=1}^N\left(P_i-\overline{P}\right)^2,$
where the sum runs over all $N$ population values.
    - Create a histogram of the population values.  Set the number of bins to match the number of distinct population values. Include error bars.

Assume $k = 0.005$ and $\beta_{\text{birth}} = 0.2$, and simulate 500,000 events.

Given a system with a constant mean population $\overline{P}$, as in the steady state above, the probability of finding the system at some time with a population of $P$ is described by the "**Poisson Distribution**":

$$
\mathcal{P}_\text{Pois.}(P) = \dfrac{\overline{P}^P}{P!}\text{e}^{-\overline{P}}.
$$

The Poisson distribution has the special property that the mean is equal to the variance $\sigma^2$,

$$
\overline{P} = \sigma^2.
$$

Overlay the Poisson distribution with a mean of $\overline{P}$ on top of your histogram.  The Poisson distribution describes probabilities, while your histogram contains counts.  Therefore, you should scale the Poisson distribution by the total number of population values $N$ in your sample.

How well does your histogram agree with the Poisson distribution?  In our simple model, are the mean and variance of the population equal, as they should be for a Poisson distribution?

You plotted the Poisson distribution using the theoretical (infinite sample) mean. With only a finite number of samples, the sample mean may not perfectly align with the theoretical mean.  Recalculate the Poisson distribution using the the sample mean and repeat the comparison with your histogram.  How does this alter the agreement?

The agreement improves, but it is still not perfect.